In [ ]:
# TemporalBench Adversarial Suite — Reversion, CausalReasoning, Interval
# Task families: Reversion, Interval, CausalReasoning, MultiReversion, IntervalMidpoint,
#                 MultiEntityJoin, FutureFact, TimelineReconstruction
# This is the hardest suite — 160 questions across 8 adversarial task families.

import json
import numpy as np
from typing import Dict, List, Optional

# @kbench.task(name="TemporalBench Adversarial", description="Reversion, CausalReasoning, Interval — 8 adversarial task families", tasks=8)

In [ ]:
class Fact:

In [ ]:
def __init__(self, key: str, value: str, valid_from: int, valid_to: int, accesses: int = 0):
        self.key = key
        self.value = value
        self.valid_from = valid_from
        self.valid_to = valid_to
        self.accesses = accesses

In [ ]:
def is_valid_at(self, day: int) -> bool:
        return self.valid_from <= day < self.valid_to

In [ ]:
def age(self, day: int) -> float:
        return day - self.valid_from

In [ ]:
def decay_score(self, day: int, half_life: float = 50.0) -> float:
        return np.exp(-0.693 * self.age(day) / half_life)

In [ ]:
def attention_score(self) -> float:
        return 1.0 + 0.001 * np.log1p(self.accesses)

In [ ]:
class TemporalAttentionStoreWithValidity:
    """System D_revised — validity windows as hard gate. Unaffected by adversarial conditions."""

In [ ]:
def __init__(self, half_life: float = 50.0):
        self.half_life = half_life
        self.facts: Dict[str, List[Fact]] = {}

In [ ]:
def put(self, key, value, valid_from, valid_to, accesses=0):
        if key not in self.facts:
            self.facts[key] = []
        self.facts[key].append(Fact(key, value, valid_from, valid_to, accesses))

In [ ]:
def get(self, key, as_of_day=None):
        if key not in self.facts:
            return None
        candidates = [f for f in self.facts[key] if f.is_valid_at(as_of_day)]
        if not candidates:
            return None
        scores = [(f.decay_score(as_of_day, self.half_life) * f.attention_score(), f) for f in candidates]
        return max(scores, key=lambda x: x[0])[1].value

In [ ]:
def get_valid_interval(self, key: str, as_of_day: int) -> Optional[tuple]:
        """Returns (valid_from, valid_to) for the fact valid at as_of_day."""
        if key not in self.facts:
            return None
        for f in self.facts[key]:
            if f.is_valid_at(as_of_day):
                return (f.valid_from, f.valid_to)
        return None

In [ ]:
def get_history(self, key: str) -> List[Fact]:
        """Return all facts for a key, sorted by valid_from."""
        if key not in self.facts:
            return []
        return sorted(self.facts[key], key=lambda f: f.valid_from)

In [ ]:
def load_adversarial(path: str) -> tuple:
    with open(path + '.jsonl') as f:
        questions = [json.loads(line) for line in f]
    return questions

In [ ]:
def score_adversarial(question: Dict, store: TemporalAttentionStoreWithValidity) -> float:
    """Score an adversarial question.
    
    Each task family requires different reasoning:
    - Reversion: fact at T-1 (prior value before current)
    - Interval: check if query day falls within a validity window
    - CausalReasoning: follow causal chain through validity windows
    - MultiReversion: multiple reverts in sequence
    - IntervalMidpoint: value at midpoint of validity window
    - MultiEntityJoin: correlate validity windows across multiple entities
    - FutureFact: detect facts not yet valid at query time
    - TimelineReconstruction: reconstruct full timeline from partial observations
    """
    task_family = question.get('task_family', 'Interval')
    query_day = question.get('query_day') or question.get('day')
    key = question.get('key', '')
    expected = question.get('expected_answer') or question.get('answer') or ''
    
    if task_family == 'Reversion':
        # "What was the value at T-1?" — need prior validity window
        history = store.get_history(key)
        for i, f in enumerate(history):
            if f.is_valid_at(query_day):
                # Found current fact — get the one before it
                if i > 0:
                    prev = history[i - 1]
                    return 1.0 if prev.value == expected else 0.0
                return 0.0  # no prior fact
        return 0.0
    
    elif task_family == 'Interval':
        # "Is day X within any validity window for key Y?"
        iv = store.get_valid_interval(key, query_day)
        if expected.lower() in ('yes', 'true', 'valid', '1'):
            return 1.0 if iv is not None else 0.0
        else:
            return 0.0 if iv is not None else 1.0
    
    elif task_family == 'CausalReasoning':
        # "Why did value X become Y at time T?" — follow causal chain
        history = store.get_history(key)
        predicted = store.get(key, query_day)
        if predicted == expected:
            return 1.0
        # Partial credit: check if we found the right causal chain
        if predicted and expected:
            pred_tokens = set(str(predicted).lower().split())
            exp_tokens = set(str(expected).lower().split())
            return len(pred_tokens & exp_tokens) / max(len(exp_tokens), 1)
        return 0.0
    
    elif task_family == 'MultiReversion':
        # Multiple reverts — check if all prior values in sequence are correct
        history = store.get_history(key)
        predicted_vals = []
        for f in history:
            if f.is_valid_at(query_day):
                predicted_vals.append(f.value)
        expected_list = expected.split('|') if '|' in expected else [expected]
        if len(predicted_vals) >= len(expected_list):
            return 1.0 if predicted_vals[-len(expected_list):] == expected_list else 0.0
        return 0.0
    
    elif task_family == 'IntervalMidpoint':
        # "What was the value at the midpoint of the validity window for key?"
        iv = store.get_valid_interval(key, query_day)
        if iv is None:
            return 0.0
        midpoint = (iv[0] + iv[1]) // 2
        predicted = store.get(key, midpoint)
        return 1.0 if predicted == expected else 0.0
    
    elif task_family == 'MultiEntityJoin':
        # Correlate validity windows across two entities
        key2 = question.get('key2', '')
        val1 = store.get(key, query_day)
        val2 = store.get(key2, query_day)
        expected_parts = expected.split('|')
        if len(expected_parts) == 2:
            return 1.0 if val1 == expected_parts[0] and val2 == expected_parts[1] else 0.0
        return 0.0
    
    elif task_family == 'FutureFact':
        # "What will be the value at day X?" — fact is not yet valid
        # A validity-aware system should detect this
        iv = store.get_valid_interval(key, query_day)
        if iv is not None:
            # Fact IS valid at query_day — this is the tricky case
            # If expected is 'none' or 'not yet', we should have returned that
            return 0.0
        # Correctly detected: fact not yet valid
        return 1.0 if expected.lower() in ('none', 'not yet', 'unknown', 'n/a') else 0.0
    
    elif task_family == 'TimelineReconstruction':
        # Reconstruct the full timeline from partial observations
        history = store.get_history(key)
        predicted = ' -> '.join([f"{f.valid_from}:{f.value}" for f in history])
        exp_tokens = set(str(expected).lower().split())
        pred_tokens = set(str(predicted).lower().split())
        overlap = len(pred_tokens & exp_tokens)
        return overlap / max(len(exp_tokens), 1)
    
    else:
        # Default: check validity interval
        predicted = store.get(key, query_day)
        return 1.0 if predicted == expected else 0.0


DATA_PATH = '/kaggle/input/temporalbench-adversarial/adversarial_temporal_questions'

print('Loading TemporalBench Adversarial Suite...')
questions = load_adversarial(DATA_PATH)
print(f'Loaded {len(questions)} adversarial questions')

# Build the store from all known facts
store = TemporalAttentionStoreWithValidity()

# We need to load facts — try the v4 facts first (most comprehensive)
try:
    import os
    for f in os.listdir('/kaggle/input'):
        import glob
        for facts_file in glob.glob(f'/kaggle/input/{f}/**/*facts*.jsonl', recursive=True):
            with open(facts_file) as ff:
                for line in ff:
                    fd = json.loads(line)
                    store.put(fd['key'], fd['value'], fd['valid_from'], fd['valid_to'],
                              fd.get('accesses', 0))
    print(f'Store populated with facts from datasets')
except Exception as e:
    print(f'Note: Could not auto-load facts ({e}). Using only embedded fact data.')

# Group by task family and score
results = {}
for q in questions:
    tf = q.get('task_family', 'Unknown')
    if tf not in results:
        results[tf] = {'correct': 0.0, 'total': 0}
    
    # Try to add fact from question if not in store
    if 'facts' in q:
        for fd in q['facts']:
            store.put(fd['key'], fd['value'], fd['valid_from'], fd['valid_to'],
                      fd.get('accesses', 0))
    
    score = score_adversarial(q, store)
    results[tf]['correct'] += score
    results[tf]['total'] += 1

print('\n--- Adversarial Suite Results ---')
metrics = {}
for tf, vals in sorted(results.items()):
    acc = vals['correct'] / vals['total'] if vals['total'] > 0 else 0.0
    metrics[tf] = acc
    print(f'  {tf:30s}: {vals["correct"]:5.1f}/{vals["total"]} = {acc:.1%}')

trs = np.mean(list(metrics.values())) if metrics else 0.0
print(f'\nAdversarial TRS: {trs:.1%}')
print('\nNote: D_revised with validity windows should score near 100% on Interval, Reversion,')
print('MultiReversion, IntervalMidpoint, and FutureFact task families.')
print('CausalReasoning and TimelineReconstruction may vary based on chain depth.')